# Backlog upload: consolidated CSVs -> Neon

One-time (or occasional) loader for pushing a full historical backlog into every Neon table. Reuses the exact same `db_neon.enforce_schema` / `db_neon.upsert_table` / `db_neon.replace_catalog_tables` helpers as the `scripts/load_*_to_neon.py` scripts — this notebook just runs all six of them, sequentially, against consolidated multi-year CSVs.

**Assumption:** each CSV referenced below already covers your *entire* backlog (every year), not just one month/year. That's what you get if you run `process_orders_to_df.py` / `process_refunds_to_df.py` / `process_labor_to_df.py` / `process_marginedge_purchaises_to_df.py` with their `*_YEAR_DIRS` env var set to a comma-separated list of *every* year folder you have, instead of just one — they already consolidate however many year folders you give them into a single CSV per table. If instead you have one CSV per year, see the note at the bottom of the Config cell.

**Safety notes:**
- Fact tables (orders/checks, items, refunds, labor, MarginEdge purchases) are loaded with `ON CONFLICT ... DO UPDATE` — safe to re-run, rows are upserted by their natural key, nothing gets duplicated.
- Catalog/dimension tables (Toast + MarginEdge catalogs) are loaded with `TRUNCATE + INSERT` (full snapshot replace) — only run those cells with a CSV that represents the *complete current* catalog, not a partial slice.
- None of these tables declare foreign keys to each other, so load order between sections doesn't matter for referential integrity.

## 1. Imports & module path

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd

# This notebook lives in scripts/, so REPO_ROOT is one level up by default.
# Running from Colab or somewhere else? Just point REPO_ROOT at wherever
# modules/db_neon.py and modules/schemas.py live, e.g.:
#   REPO_ROOT = Path("/content/drive/MyDrive/Data Projects/restaurant-analytics")
REPO_ROOT = Path.cwd().parent
MODULE_DIR = REPO_ROOT / "modules"
sys.path.insert(0, str(MODULE_DIR))

import db_neon
import schemas

print(f"REPO_ROOT:  {REPO_ROOT}")
print(f"MODULE_DIR: {MODULE_DIR}  (exists: {MODULE_DIR.exists()})")

## 2. Neon connection

Pasted via a hidden prompt rather than hardcoded — this notebook may end up committed to git, and a connection string in a code cell would leak your DB credentials into the repo's history.

In [ ]:
import getpass

os.environ["NEON_CONNECTION_STRING"] = getpass.getpass("Neon connection string: ")

engine = db_neon.get_neon_engine()
print("Neon engine created.")

## 3. Config: where your consolidated backlog CSVs live

Edit these paths to point at your actual backlog output folders. Defaults match the same relative folders (and filenames) the `process_*_to_df.py` / `load_*_to_neon.py` scripts already use.

**If you have one CSV per year instead of one combined CSV per table:** concatenate them before calling the loader, e.g. `pd.concat([pd.read_csv(p) for p in sorted(dir.glob("toast_order_checks_*.csv"))])`, and pass that combined DataFrame in — or just call `load_and_upsert` once per year-file back to back; `ON CONFLICT DO UPDATE` makes repeated calls against the same table safe regardless of order.

In [ ]:
BACKLOG_DIRS = {
    "orders":               REPO_ROOT / "raw_orders",              # toast_order_checks.csv, item_sales.csv
    "refunds":               REPO_ROOT / "raw_refunds",             # refunds_daily_summary.csv
    "labor":                 REPO_ROOT / "raw_labor",               # labor_time_entries.csv
    "marginedge":            REPO_ROOT / "raw_marginedge",          # marginedge_orders.csv, marginedge_line_items.csv
    "toast_catalogs":        REPO_ROOT / "catalogs",                # catalog_*.csv
    "marginedge_catalogs":   REPO_ROOT / "catalogs" / "marginedge", # catalog_categories.csv, catalog_vendors.csv
}

for name, path in BACKLOG_DIRS.items():
    print(f"{name:22s} -> {path}  (exists: {path.exists()})")

## 4. Helpers

Thin wrappers around `db_neon`, identical in behavior to what every `load_*_to_neon.py` script already does — read CSV, enforce the schema (dtypes + column selection, per `schemas.py`), then either upsert (fact tables) or truncate+replace (catalog/dimension snapshots).

In [ ]:
def load_and_upsert(engine, table_name: str, csv_path: Path, schema: dict, key_columns: list[str]) -> int:
    if not csv_path.exists():
        print(f"[{table_name}] SKIPPED: {csv_path} not found.")
        return 0
    df = pd.read_csv(csv_path)
    df = db_neon.enforce_schema(df, schema)
    return db_neon.upsert_table(engine=engine, table_name=table_name, df=df, key_columns=key_columns)


def load_and_replace(engine, catalog_dir: Path, table_map: dict) -> dict:
    table_to_df = {}
    for table_name, config in table_map.items():
        csv_path = catalog_dir / config["csv"]
        if not csv_path.exists():
            print(f"[{table_name}] SKIPPED: {csv_path} not found.")
            continue
        df = pd.read_csv(csv_path)
        table_to_df[table_name] = db_neon.enforce_schema(df, config["schema"])
    return db_neon.replace_catalog_tables(engine, table_to_df)

## 5. Toast orders & items (`toast_order_checks`, `item_sales`)
Upsert, keyed on `check_guid` / `selection_guid`.

In [ ]:
load_and_upsert(engine, "toast_order_checks", BACKLOG_DIRS["orders"] / "toast_order_checks.csv",
                schemas.TOAST_ORDER_CHECKS, ["check_guid"])

load_and_upsert(engine, "item_sales", BACKLOG_DIRS["orders"] / "item_sales.csv",
                schemas.TOAST_ITEM_SALES, ["selection_guid"])

## 6. Toast refunds (`refunds_daily_summary`)
Upsert, keyed on `guid` (this table is per-refund-transaction grain, despite the name).

In [ ]:
load_and_upsert(engine, "refunds_daily_summary", BACKLOG_DIRS["refunds"] / "refunds_daily_summary.csv",
                schemas.TOAST_REFUND_DAILY_SUMMARY, ["guid"])

## 7. Toast labor (`labor_time_entries`)
Upsert, keyed on `time_entry_guid`.

In [ ]:
load_and_upsert(engine, "labor_time_entries", BACKLOG_DIRS["labor"] / "labor_time_entries.csv",
                schemas.TOAST_LABOR_TIME_ENTRIES, ["time_entry_guid"])

## 8. MarginEdge purchases (`marginedge_orders`, `marginedge_line_items`)
Upsert, keyed on `order_id` / `(order_id, line_item_index)`.

In [ ]:
load_and_upsert(engine, "marginedge_orders", BACKLOG_DIRS["marginedge"] / "marginedge_orders.csv",
                schemas.MARGINEDGE_ORDERS_SCHEMA, ["order_id"])

load_and_upsert(engine, "marginedge_line_items", BACKLOG_DIRS["marginedge"] / "marginedge_line_items.csv",
                schemas.MARGINEDGE_LINE_ITEMS_SCHEMA, ["order_id", "line_item_index"])

## 9. Toast catalogs (full snapshot replace)
`catalog_tables`, `catalog_dining_options`, `catalog_menu_items`, `catalog_revenue_centers`, `catalog_sales_categories`.

In [ ]:
TOAST_CATALOG_MAP = {
    "catalog_tables":           {"csv": "catalog_tables.csv",           "schema": schemas.TOAST_CATALOG_TABLES},
    "catalog_dining_options":   {"csv": "catalog_dining_options.csv",   "schema": schemas.TOAST_CATALOG_DINING_OPTIONS},
    "catalog_menu_items":       {"csv": "catalog_menu_items.csv",       "schema": schemas.TOAST_CATALOG_MENU_ITEMS},
    "catalog_revenue_centers":  {"csv": "catalog_revenue_centers.csv",  "schema": schemas.TOAST_CATALOG_REVENUE_CENTERS},
    "catalog_sales_categories": {"csv": "catalog_sales_categories.csv", "schema": schemas.TOAST_CATALOG_SALES_CATEGORIES},
}

results = load_and_replace(engine, BACKLOG_DIRS["toast_catalogs"], TOAST_CATALOG_MAP)
print(results)

## 10. MarginEdge catalogs (full snapshot replace)
`catalog_purchase_categories`, `catalog_vendors`.

In [ ]:
MARGINEDGE_CATALOG_MAP = {
    "catalog_purchase_categories": {"csv": "catalog_categories.csv", "schema": schemas.MARGINEDGE_CATALOG_PURCHAISE_CATEGORIES},
    "catalog_vendors":             {"csv": "catalog_vendors.csv",    "schema": schemas.MARGINEDGE_CATALOG_VENDORS_SCHEMA},
}

results = load_and_replace(engine, BACKLOG_DIRS["marginedge_catalogs"], MARGINEDGE_CATALOG_MAP)
print(results)

## 11. Verify: row counts per table
Quick sanity check that everything actually landed.

In [ ]:
from sqlalchemy import text

ALL_TABLES = [
    "toast_order_checks", "item_sales", "refunds_daily_summary", "labor_time_entries",
    "marginedge_orders", "marginedge_line_items",
    "catalog_tables", "catalog_dining_options", "catalog_menu_items",
    "catalog_revenue_centers", "catalog_sales_categories",
    "catalog_purchase_categories", "catalog_vendors",
]

with engine.connect() as conn:
    for table in ALL_TABLES:
        count = conn.execute(text(f'SELECT COUNT(*) FROM "{table}"')).scalar()
        print(f"{table:28s} {count:>10,} rows")